# NEREID-B — GPU (CuPy) verification on Google Colab

Verifies the CuPy GPU port of `solver.py` on a real CUDA device. The CPU path is the
validated default; the GPU port is designed to be numerically identical (same IEEE-double
kernels), so a correct port differs only by floating-point reduction order (~1e-9).

**Rev 2.0** added an **on-device sparse pressure solve** (`--gpu-poisson`): the Poisson
solve can now stay on the GPU (no per-step host↔device round-trip), for BOTH the
free-surface AND rigid-lid cases. `--gpu-poisson-direct` factorises the SPD operator ONCE
on the device and reuses it (single-solve speedup), falling back to warm-started PCG if the
installed CuPy has no device sparse factoriser. This notebook run-verifies that path — the
one check that needed a live CUDA device.

**Before running:** Runtime ▸ Change runtime type ▸ **GPU** (T4 is enough).

Then get `solver.py` here (clone your repo branch, or Files ▸ upload).

In [ ]:
# 1. Confirm the GPU is attached
!nvidia-smi

In [ ]:
# 2. Pin the dependency stack.  CRITICAL: numpy MUST be 1.26.4 — numpy 2.x breaks
#    scipy 1.11.4's sparse import and kills the pressure solver.  cupy-cuda12x matches
#    Colab's CUDA 12 runtime (use cupy-cuda11x if nvidia-smi shows CUDA 11).
!pip -q install numpy==1.26.4 scipy==1.11.4 matplotlib
!pip -q install cupy-cuda12x
# gsw is OPTIONAL (TEOS-10 eos_mode); it declares numpy>=2 but works on 1.26.4.
# Install it ONLY if you need eos_mode="teos10"; otherwise skip to keep numpy pinned.
# !pip -q install --no-deps gsw
print('IMPORTANT: Runtime ▸ Restart session now, then run the cells below (do NOT re-run this cell).')

In [ ]:
# 3. (after restart) sanity-check the versions
import numpy, scipy, cupy
print('numpy', numpy.__version__, '(must be 1.26.4)')
print('scipy', scipy.__version__)
print('cupy ', cupy.__version__)
print('CUDA devices', cupy.cuda.runtime.getDeviceCount())

In [ ]:
# 4. Make sure solver.py is here (clone your repo branch, or upload via the Files panel).
#    e.g.  !git clone --branch solidify-solver https://github.com/<you>/<repo>.git nereid
#          %cd nereid
import os; assert os.path.exists('solver.py'), 'Get solver.py into the Colab working dir first.'
print('solver.py found:', os.path.abspath('solver.py'))

In [ ]:
# 5. Device status
!python solver.py --gpu-check

In [ ]:
# 6a. Base port verification: same short sim on CPU and GPU, report max|CPU-GPU|.
#     Pressure solve on the host LU in BOTH runs -> PASS band rel < 1e-6 (float round-off).
!python solver.py --gpu-verify

In [ ]:
# 6b. Rev 2.0 ON-DEVICE POISSON verification: the GPU run ALSO solves the pressure on-device.
#     Matches the host LU only to the CG tolerance (~1e-10), so the PASS band is rel < 1e-4.
#     --gpu-poisson-direct adds the factorise-once device LU (falls back to device PCG if the
#     installed CuPy lacks a sparse factoriser -- still a valid PASS).
!python solver.py --gpu-verify --gpu-poisson                     # warm-started device PCG
!python solver.py --gpu-verify --gpu-poisson --gpu-poisson-direct # factorise-once device LU

In [ ]:
# 7. Confirm the invariants still hold on this machine (CPU backend), then run a real
#    prediction on the GPU with the pressure solve on-device too.
!python solver.py --selftest
!python solver.py --gpu --gpu-poisson --nx 64 --ny 40 --nz 28 --t_end 360

In [ ]:
# 8. (Rev 2.0, #7) RESOLVED NEAR-FIELD on the GPU: auto fine-mesh two-way nest so the raw
#    resolved jet (near_field_coupling=False) no longer over-predicts on a coarse grid.
#    Heavy at high refine -> exactly what the GPU is for. Raise resolved_nf_max_refine via a
#    --config JSON to push the child finer once --gpu-verify is green.
!python solver.py --resolved-nearfield --gpu --gpu-poisson

## Interpreting the result

- `--gpu-verify` **VERIFIED** (rel diffs ~1e-9 to 1e-6 per field) → the CuPy port is correct;
  the GPU path can be used for the large/fine-grid runs the CPU can't afford.
- `--gpu-verify --gpu-poisson` **VERIFIED** (rel < 1e-4) → the on-device pressure solve
  reproduces the host LU to the CG tolerance; the Poisson solve can stay on the GPU (no
  per-step host round-trip). Add `--gpu-poisson-direct` to also verify the factorise-once
  device LU (or its clean PCG fallback).
- A **MISMATCH** (rel diff ≫ the band) points at a host↔device boundary that still leaks a
  CuPy array into a host (SciPy/`np.add.at`) kernel — report the field name that failed.
- This is a **one-time** trust check per environment, not a per-run prerequisite. Once it is
  VERIFIED here, go straight to `--gpu --gpu-poisson` simulations; re-verify only if you
  change the CUDA/CuPy/numpy stack, the GPU type, or any GPU code path in `solver.py`. CPU
  runs never need it.